In [ ]:
import pandas as pd
import numpy as np
import string
from scipy import stats

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression
from nltk.stem import SnowballStemmer

from xgboost import XGBRegressor

In [ ]:
stop_words = set(ENGLISH_STOP_WORDS)

In [ ]:
def spearman_corr(y_true, y_pred):

    scores = []

    for i in range(y_true.shape[1]):
        corr = stats.spearmanr(y_true[:, i], y_pred[:, i]).correlation
        scores.append(corr)

    return np.nanmean(scores)

In [ ]:
train = pd.read_csv(
    "/kaggle/input/competitions/google-quest-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/google-quest-challenge/test.csv"
)

sample_sub = pd.read_csv(
    "/kaggle/input/competitions/google-quest-challenge/sample_submission.csv"
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

all_text = pd.concat([
    train["question_title"],
    train["question_body"],
    train["answer"],
    test["question_title"],
    test["question_body"],
    test["answer"]
])

tfidf.fit(all_text)

In [ ]:
stemmer = SnowballStemmer("english")

def stem_tokenizer(text):
    return [stemmer.stem(w) for w in str(text).lower().split()]

In [ ]:
def extract_features(df, tfidf):

    df = df.copy()

    for col in ["question_title","question_body","answer"]:

        df[f"{col}_word_len"] = df[col].fillna("").apply(
            lambda x: len(x.split())
        )

        df[f"{col}_unique_ratio"] = df[col].fillna("").apply(
            lambda x: len(set(x.split())) / (len(x.split()) + 1)
        )

        df[f"{col}_char_len"] = df[col].fillna("").apply(len)

        df[f"{col}_punct"] = df[col].apply(
            lambda x: sum(c in string.punctuation for c in str(x))
        )

    # QA word overlap
    def clean_words(text):
        return [
            w for w in str(text).lower().split()
            if w not in stop_words and w not in string.punctuation
        ]

    q_words = (df["question_title"] + " " + df["question_body"]).apply(clean_words)
    a_words = df["answer"].apply(clean_words)

    df["qa_overlap"] = [
        len(set(q) & set(a)) for q, a in zip(q_words, a_words)
    ]

    # TFIDF cosine similarities
    title_vec = tfidf.transform(df["question_title"])
    body_vec  = tfidf.transform(df["question_body"])
    answer_vec = tfidf.transform(df["answer"])

    df["title_answer_cos"] = cosine_similarity(title_vec, answer_vec).diagonal()
    df["body_answer_cos"]  = cosine_similarity(body_vec, answer_vec).diagonal()
    df["title_body_cos"]   = cosine_similarity(title_vec, body_vec).diagonal()

    return df

In [ ]:
train = extract_features(train, tfidf)
test = extract_features(test, tfidf)

In [ ]:
target_cols = [
    c for c in train.columns
    if c not in test.columns and c != "qa_id"
]

y = train[target_cols].values

# valid labels for each target (used for nearest-label snapping)
label_values = {
    col: np.sort(train[col].unique())
    for col in target_cols
}


In [ ]:
num_cols = [
    "question_title_word_len",
    "question_body_word_len",
    "answer_word_len",
    "question_title_unique_ratio",
    "question_body_unique_ratio",
    "answer_unique_ratio",
    "qa_overlap"
]

cat_cols = [
    "category",
    "host"
]

text_cols = [
    "question_title",
    "question_body",
    "answer"
]

In [ ]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "title",
            TfidfVectorizer(
                max_features=5000,
                ngram_range=(1,2),
                tokenizer=stem_tokenizer,
                token_pattern=None
            ),
            "question_title"
        ),

        (
            "body",
            TfidfVectorizer(
                max_features=10000,
                tokenizer=stem_tokenizer,
                token_pattern=None
            ),
            "question_body"
        ),

        (
            "answer",
            TfidfVectorizer(
                max_features=10000,
                tokenizer=stem_tokenizer,
                token_pattern=None
            ),
            "answer"
        ),

        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("power", PowerTransformer())
            ]),
            num_cols
        ),

        (
            "cat",
            Pipeline([
                ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                ("ohe", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

In [ ]:
xtrain = train
xtest = test

ytrain = train[target_cols].values

In [ ]:
n_folds = 5

gkf = GroupKFold(n_splits=n_folds)

oof = np.zeros(y.shape)

test_preds = np.zeros((len(test), len(target_cols)))

groups = train["question_body"]

In [ ]:
alphas = [10,20,30]

def build_target_model(alpha):

    # Keep Ridge for lower penalties; use ElasticNet for higher penalties
    # with lighter effective shrinkage.
    if alpha >= 20:
        return ElasticNet(
            alpha=alpha / 10,
            l1_ratio=0.05,
            max_iter=5000,
            random_state=42
        )

    return Ridge(alpha=alpha)

def snap_to_known_labels(preds, target_cols, label_values):

    snapped = np.zeros_like(preds)

    for j, col in enumerate(target_cols):
        vals = label_values[col]
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        snapped[:, j] = vals[idx]

    return snapped

best_score = -1
best_alpha = None
best_models = None
best_oof = None
best_oof_snap = None
best_test = None
best_test_snap = None

for alpha in alphas:

    print(f"\nTesting alpha = {alpha}")

    feature_pipe = Pipeline([
        ("prep", preprocessor),
        ("svd", TruncatedSVD(n_components=400, random_state=42)),
        ("scale", StandardScaler())
    ])

    models = []
    oof = np.zeros((len(xtrain), len(target_cols)))
    test_preds = np.zeros((len(xtest), len(target_cols)))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(xtrain, ytrain, groups)):

        print("Fold", fold + 1)

        X_tr = xtrain.iloc[tr_idx]
        X_val = xtrain.iloc[val_idx]

        y_tr = ytrain[tr_idx]

        fold_feature_pipe = clone(feature_pipe)
        X_tr_feat = fold_feature_pipe.fit_transform(X_tr)
        X_val_feat = fold_feature_pipe.transform(X_val)
        X_test_feat = fold_feature_pipe.transform(xtest)

        fold_models = {}

        for target_idx, target_name in enumerate(target_cols):

            reg = build_target_model(alpha)
            reg.fit(X_tr_feat, y_tr[:, target_idx])

            oof[val_idx, target_idx] = reg.predict(X_val_feat)
            test_preds[:, target_idx] += reg.predict(X_test_feat) / n_folds

            fold_models[target_name] = reg

        models.append({
            "features": fold_feature_pipe,
            "targets": fold_models
        })

    oof_snap = snap_to_known_labels(oof, target_cols, label_values)
    test_snap = snap_to_known_labels(test_preds, target_cols, label_values)

    score_raw = spearman_corr(ytrain, oof)
    score_snap = spearman_corr(ytrain, oof_snap)
    score = max(score_raw, score_snap)

    print("CV Spearman (raw):", score_raw)
    print("CV Spearman (snapped):", score_snap)

    if score > best_score:
        best_score = score
        best_alpha = alpha
        best_models = models
        best_oof = oof.copy()
        best_oof_snap = oof_snap.copy()
        best_test = test_preds.copy()
        best_test_snap = test_snap.copy()

print("\nBest alpha:", best_alpha)
print("Best CV:", best_score)

models = best_models
oof = best_oof
oof_snap = best_oof_snap
test_preds = best_test
test_snap = best_test_snap


In [ ]:
calibrators = []

for i in range(len(target_cols)):

    ir = IsotonicRegression(
        y_min=0,
        y_max=1,
        out_of_bounds="clip"
    )

    ir.fit(oof[:, i], ytrain[:, i])

    calibrators.append(ir)

In [ ]:
oof_cal = np.zeros_like(oof)

for i, ir in enumerate(calibrators):
    oof_cal[:, i] = ir.transform(oof[:, i])

print("Calibrated CV (raw):", spearman_corr(ytrain, oof_cal))
print("Snapped CV:", spearman_corr(ytrain, oof_snap))


In [ ]:
test_cal = np.zeros_like(test_preds)

for i, ir in enumerate(calibrators):
    test_cal[:, i] = ir.transform(test_preds[:, i])

test_cal_snap = snap_to_known_labels(test_cal, target_cols, label_values)


In [ ]:
def rank_normalize(arr):

    ranks = arr.argsort(axis=0).argsort(axis=0)

    ranks = ranks / ranks.max(axis=0)

    return ranks

test_rank = rank_normalize(test_preds)
test_snap_rank = rank_normalize(test_snap)
test_cal_rank = rank_normalize(test_cal)
test_cal_snap_rank = rank_normalize(test_cal_snap)


In [ ]:
test_final = (
    0.25 * test_rank +
    0.25 * test_snap_rank +
    0.25 * test_cal_rank +
    0.25 * test_cal_snap_rank
)


In [ ]:
submission = sample_sub.copy()

submission[target_cols] = np.clip(test_final, 0, 1)

submission.to_csv("submission.csv", index=False)

print("Submission saved!")

In [ ]:
print(submission.shape)
print(submission.isna().sum().sum())